In [7]:
from skills.co_occur import calculate_node_alarm_weights 
from SkillExecutor import SkillExecutor

import os
import json



def scan_co_occurrence_hits(nodes_dir: str, weight_path: str, co_occur_path: str):
    print(f"🔍 启动探雷扫描...\n扫描目录: {nodes_dir}\n经验库: {co_occur_path}")
    
    if not os.path.exists(co_occur_path):
        print(f"❌ 错误: 找不到经验库文件 {co_occur_path}，请确认是否已生成！")
        return

    hit_cases = []
    miss_count = 0
    error_count = 0

    # 遍历你清洗后的所有 nodes 文件 (例如: /home/sbp/lixinyang/pingmesh/data/nodes)
    for root, dirs, files in os.walk(nodes_dir):
        for file in files:
            # 只处理 nodes 的 JSON 文件，避开 info.json
            if file.endswith(".json") and "info" not in file:
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        node_data = json.load(f)
                        
                    # 兼容格式：如果是字典，转成列表；如果是列表，直接用
                    node_list = []
                    if isinstance(node_data, dict):
                        node_list = list(node_data.values())
                    elif isinstance(node_data, list):
                        node_list = node_data
                        
                    # ==========================================
                    # 执行 Skill 1 进行摸底测试
                    # ==========================================
                    result_str = calculate_node_alarm_weights(
                        node_list=node_list,
                        dirpath=weight_path,
                        co_occur_path=co_occur_path
                    )
                    
                    # 检查是否命中了我们在 Skill 1 里埋下的特权警告标识
                    if "🚨 【系统级高危告警组合拦截 (由历史错案反思生成)】 🚨" in result_str:
                        # 截取大模型将要看到的 Warning 文本
                        warning_snippet = result_str.split("==================================================")[-2].strip()
                        hit_cases.append({
                            "path": file_path,
                            "warning": warning_snippet
                        })
                    else:
                        miss_count += 1
                        
                except Exception as e:
                    error_count += 1
                    # print(f"读取或执行文件 {file_path} 失败: {e}")

    # ================= 输出统计报告 =================
    print("\n" + "="*50)
    print("📊 告警共现规则命中摸底报告")
    print("="*50)
    total_cases = len(hit_cases) + miss_count + error_count
    print(f"总计扫描 Case 数量 : {total_cases}")
    print(f"✅ 成功命中规则的 Case: {len(hit_cases)} (占比 {len(hit_cases)/total_cases*100:.2f}%)")
    print(f"⚪ 未命中任何规则 Case: {miss_count}")
    print(f"❌ 解析失败异常 Case  : {error_count}")
    print("="*50)

    if hit_cases:
        print("\n👇 命中详情抽样 (前 5 个) 👇")
        for i, item in enumerate(hit_cases[:5]):
            print(f"【Case {i+1}】: {item['path']}")
            print(f"{item['warning']}")
            print("-" * 50)
            
        #如果你想看全量，可以把下面这行解开
        with open("hit_report.json", "w", encoding="utf-8") as f:
            json.dump(hit_cases, f, ensure_ascii=False, indent=2)



In [10]:

# 配置你的路径
NODES_DIR = "/home/sbp/lixinyang/pingmesh/data/nodes"  # 你的原始/清洗后节点目录
WEIGHT_PATH = "/home/sbp/lixinyang/pingmesh/data/weights/classified_alarms/all_alarms.json"
CO_OCCUR_PATH = "/home/sbp/lixinyang/pingmesh/SkillBank/alarm_co_occurrence_rules.json"

scan_co_occurrence_hits(NODES_DIR, WEIGHT_PATH, CO_OCCUR_PATH)

🔍 启动探雷扫描...
扫描目录: /home/sbp/lixinyang/pingmesh/data/nodes
经验库: /home/sbp/lixinyang/pingmesh/SkillBank/alarm_co_occurrence_rules.json

📊 告警共现规则命中摸底报告
总计扫描 Case 数量 : 416
✅ 成功命中规则的 Case: 123 (占比 29.57%)
⚪ 未命中任何规则 Case: 293
❌ 解析失败异常 Case  : 0

👇 命中详情抽样 (前 5 个) 👇
【Case 1】: /home/sbp/lixinyang/pingmesh/data/nodes/1761019500000/511425852/nodes.json
🚨 【系统级高危告警组合拦截 (由历史错案反思生成)】 🚨
（说明：大模型必须严格遵守以下特权指令，否则会导致严重误判！）
⚠️ 【专家指令 - CO_OCCUR_HWLLDP_LLDPCHANGE】检测到高危组合爆发：['hwLldpInterfaceRemTablesChange', 'linkDown_active'] 
👉 避坑指南: 必须检查所有节点中带有 hwLldpInterfaceRemTablesChange 和 linkDown_active 告警的设备，即使它们的综合嫌疑度较低，也不能排除它们作为根因的可能性。这些设备的告警组合必须被视为关键指标。

⚠️ 【专家指令 - CO_OCCUR_001】检测到高危组合爆发：['RM_DELETE_DEFAULTRT', 'hwLocalFaultAlarm_active_for_CND', 'QOS_PACKET_DROP'] 
👉 避坑指南: None

⚠️ 【专家指令 - CO_OCCUR_NETWORK_FAULT】检测到高危组合爆发：['RM_DELETE_DEFAULTRT', 'QOS_PACKET_DROP', 'hwLocalFaultAlarm_active_for_CND'] 
👉 避坑指南: 当检测到RM_DELETE_DEFAULTRT或hwLocalFaultAlarm_active_for_CND时，必须同时检查是否存在并发的QOS_PACKET_DROP告警，以防止漏报真实根因。严禁仅依赖单一